In [ ]:
from __future__ import annotations
from pathlib import Path
from dataclasses import dataclass
import math
import zipfile
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.patches import FancyBboxPatch, FancyArrowPatch

In [ ]:
se

In [1]:
!python --version

Python 3.12.13


In [ ]:
# ---------------------------------------------------------------------
# Reproducibility and output location
# ---------------------------------------------------------------------

SEED = 12345
random.seed(SEED)
np.random.seed(SEED)

# ---------------------------------------------------------------------
# Basic helpers
# ---------------------------------------------------------------------

def simpson_wcc(n_cycles: int) -> np.ndarray:
    weights = []
    for i in range(n_cycles + 1):
        if i == 0 or i == n_cycles:
            weights.append(1.0 / 3.0)
        elif i % 2 == 1:
            weights.append(4.0 / 3.0)
        else:
            weights.append(2.0 / 3.0)
    return np.array(weights, dtype=float)


def prob_to_rate(p: float) -> float:
    p = float(np.clip(p, 1e-12, 1.0 - 1e-12))
    return -np.log(1.0 - p)


def rates_to_row_probs(rates: dict[str, float], dt: float) -> dict[str, float]:
    total_rate = float(sum(rates.values()))
    if total_rate <= 0:
        out = {k: 0.0 for k in rates}
        out["stay"] = 1.0
        return out
    move_prob = 1.0 - np.exp(-total_rate * dt)
    row = {k: (rates[k] / total_rate) * move_prob for k in rates}
    row["stay"] = 1.0 - move_prob
    return row


def clamp01(x: float) -> float:
    return max(0.0, min(1.0, float(x)))


def safe_icer(cost: float, effect: float) -> float:
    return np.nan if effect <= 0 else float(cost) / float(effect)


def beta_from_mean_k(mean: float, k: float, eps: float = 1e-9) -> tuple[float, float]:
    m = min(1.0 - eps, max(eps, float(mean)))
    a = max(m * k, eps)
    b = max((1.0 - m) * k, eps)
    return a, b


def truncnorm_sample(mean: float, sd: float, low: float, high: float, rng: np.random.Generator) -> float:
    if sd <= 0:
        return float(np.clip(mean, low, high))
    for _ in range(10000):
        x = float(rng.normal(mean, sd))
        if low <= x <= high:
            return x
    return float(np.clip(mean, low, high))


def gamma_multiplier(mean: float, cv: float, rng: np.random.Generator) -> float:
    shape = 1.0 / (cv ** 2)
    scale = mean / shape
    return float(rng.gamma(shape, scale))


def wilson_ci(k: int, n: int, z: float = 1.959963984540054) -> tuple[float, float]:
    if n <= 0:
        return (np.nan, np.nan)
    p = k / n
    denom = 1.0 + z ** 2 / n
    centre = (p + z ** 2 / (2 * n)) / denom
    half = z * math.sqrt((p * (1 - p) + z ** 2 / (4 * n)) / n) / denom
    return max(0.0, centre - half), min(1.0, centre + half)


def quantile_summary(values: np.ndarray) -> tuple[float, float, float]:
    vals = np.asarray(values, dtype=float)
    vals = vals[np.isfinite(vals)]
    if vals.size == 0:
        return (np.nan, np.nan, np.nan)
    return float(np.mean(vals)), float(np.percentile(vals, 2.5)), float(np.percentile(vals, 97.5))


def find_wtp_at_probability(wtp_thresholds: np.ndarray, ceac: np.ndarray, probability: float) -> float:
    ceac = np.asarray(ceac, dtype=float)
    idx = np.where(ceac >= probability)[0]
    if idx.size == 0:
        return np.nan
    i = int(idx[0])
    if i == 0:
        return float(wtp_thresholds[0])
    x0, x1 = float(wtp_thresholds[i - 1]), float(wtp_thresholds[i])
    y0, y1 = float(ceac[i - 1]), float(ceac[i])
    if y1 == y0:
        return x1
    return float(x0 + (probability - y0) * (x1 - x0) / (y1 - y0))


In [ ]:

# ---------------------------------------------------------------------
# Cost Inputs from three cost scenarios
# ---------------------------------------------------------------------

# Cost inputs (2024 USD)
program_manager = 491.34 * 12
logistics_super = 640.85 * 12
mne_officer = 160.00 * 12
staff_support = 932.91 * 12
other_direct = 330.85 * 12

startup_field_travel = 7_862.66
legacy_startup_2022 = 20_000.00
comm_materials = 2_500.00

training_initial = 8_000.00
training_refresh = 4_000.00

drone1_rental = 115_200.00
drone2_rental = 80_000.00
takeoff_point = 60_000.00
packing_material = 13_200.00

minsan_local = 5_000.00
supervision_visits = 28_966.67

km_avoided_y1 = 45_000
cost_per_km = 1.26
wastage_saved_y1 = 12_500
labour_hours_saved = 3_200
hourly_wage = 1.50

offsets_y1 = (
    km_avoided_y1 * cost_per_km
    + wastage_saved_y1
    + labour_hours_saved * hourly_wage
)

base_program_admin_y1 = program_manager + logistics_super + staff_support + other_direct
base_monitoring_y1 = mne_officer + minsan_local + supervision_visits
base_startup_y1 = startup_field_travel + legacy_startup_2022 + comm_materials
base_training_y1 = training_initial + training_refresh

observed_drone_month_fee = (drone1_rental + drone2_rental) / (12 + 10)
observed_top_month_fee = takeoff_point / 12.0
packing_cost_per_delivery = packing_material / 1760.0

mixed_drone_month_fee = 7_500.00
mixed_overage_fee = 110.00
mixed_threshold_flights = 75.0

at_scale_drone_month_fee = 4_800.00
at_scale_top_month_fee = 3_600.00

YEARS = 10
DISCOUNT_RATE = 0.03
cycle_length = 1.0 / 12.0
n_cycles = YEARS * 12
H, U, S, D = 0, 1, 2, 3

COMMON_DRONES_BY_YEAR = np.array([2, 4, 6, 8, 10, 12, 14, 16, 18, 20], dtype=float)
COMMON_TOPS_BY_YEAR = np.array([1, 2, 3, 4, 5, 6, 7, 8, 9, 10], dtype=float)

SCENARIOS = [
    {
        "Scenario_ID": 1,
        "Scenario": "Expensive pilot pricing + underused network",
        "Price_regime": "Observed pilot/FID pricing",
        "Support_model": "Pilot-style support",
        "Description": (
            "Observed pilot pricing is retained and the network expands, "
            "but realised flights remain low, so fixed drone and site costs "
            "are spread across too few deliveries."
        ),
        "drone_month_fee": observed_drone_month_fee,
        "top_month_fee": observed_top_month_fee,
        "overage_fee": 0.0,
        "overage_threshold": mixed_threshold_flights,
        "flights_per_drone_month_year2plus": 55.0,
        "support_multiplier_year2plus": 1.00,
    },
    {
        "Scenario_ID": 2,
        "Scenario": "Transitional GAVI pricing + expected utilisation",
        "Price_regime": "Transitional mixed/GAVI pricing",
        "Support_model": "Transition to routine delivery",
        "Description": (
            "Partner-described GAVI mixed pricing is used: US$7,500 per "
            "drone-month plus US$110 for each flight above 75 per drone-month. "
            "The network reaches the expected utilisation of about 80 flights "
            "per drone-month. This is the PSA base case."
        ),
        "drone_month_fee": mixed_drone_month_fee,
        "top_month_fee": observed_top_month_fee,
        "overage_fee": mixed_overage_fee,
        "overage_threshold": mixed_threshold_flights,
        "flights_per_drone_month_year2plus": 80.0,
        "support_multiplier_year2plus": 0.85,
    },
    {
        "Scenario_ID": 3,
        "Scenario": "At-scale pricing + efficient scale-up",
        "Price_regime": "Negotiated at-scale pricing",
        "Support_model": "Routine implementation",
        "Description": (
            "Lower at-scale prices are achieved because the programme scales "
            "efficiently and contracts become more stable. The model uses "
            "US$4,800 per drone-month, US$3,600 per TOP-month, and 90 flights "
            "per drone-month."
        ),
        "drone_month_fee": at_scale_drone_month_fee,
        "top_month_fee": at_scale_top_month_fee,
        "overage_fee": 0.0,
        "overage_threshold": mixed_threshold_flights,
        "flights_per_drone_month_year2plus": 90.0,
        "support_multiplier_year2plus": 0.70,
    },
]
SCENARIO_TABLE = pd.DataFrame(SCENARIOS)

In [ ]:
# Population and natural history
facilities_drone = 54
catchment_per_facility = 9_000
u5_share = 0.18
initial_u5_population = facilities_drone * catchment_per_facility * u5_share

total_population = facilities_drone * catchment_per_facility
crude_birth_rate_per_1000 = 30.0
births_per_month = (crude_birth_rate_per_1000 / 1000.0) * total_population / 12.0

BASE_PARAMS = {
    "p_HD": 0.007,
    "p_HU": 0.40,
    "p_UH": 0.861,
    "p_US": 0.13,
    "p_UD": 0.009,
    "p_SH": 0.938,
    "p_SD": 0.062,
    "u_U": 0.006,
    "u_S": 0.133,
    "LE0": 62.0,
    "control_rdt_in_stock": 0.81,
    "delta_rdt_in_stock": 0.15,
    "control_test_if_stock": 0.60,
    "delta_test_if_stock": 0.19,
    "control_act_if_positive": 0.82,
    "pass_through": 0.82,
    "medical_cost_scale": 1.0,
    "act_eff": 0.96,
}

PSA_SPECS = {
    "Clinical probabilities": [
        {"name": "p_HD", "base": 0.007, "distribution": "Beta(mean, k=100)", "range": "derived from distribution", "source": "Tightened from supplementary methods"},
        {"name": "p_HU", "base": 0.400, "distribution": "Beta(mean, k=100)", "range": "derived from distribution", "source": "Tightened from supplementary methods"},
        {"name": "p_UH", "base": 0.861, "distribution": "Beta(mean, k=100)", "range": "derived from distribution", "source": "Tightened from supplementary methods"},
        {"name": "p_US", "base": 0.130, "distribution": "Beta(mean, k=100)", "range": "derived from distribution", "source": "Tightened from supplementary methods"},
        {"name": "p_UD", "base": 0.009, "distribution": "Beta(mean, k=100)", "range": "derived from distribution", "source": "Tightened from supplementary methods"},
        {"name": "p_SH", "base": 0.938, "distribution": "Beta(mean, k=100)", "range": "derived from distribution", "source": "Tightened from supplementary methods"},
        {"name": "p_SD", "base": 0.062, "distribution": "Beta(mean, k=100)", "range": "derived from distribution", "source": "Tightened from supplementary methods"},
    ],
    "Disability weights": [
        {"name": "u_U", "base": 0.006, "distribution": "Beta(mean, k=100)", "range": "derived from distribution", "source": "Tightened from supplementary methods"},
        {"name": "u_S", "base": 0.133, "distribution": "Beta(mean, k=100)", "range": "derived from distribution", "source": "Tightened from supplementary methods"},
    ],
    "RCT-derived effectiveness deltas": [
        {"name": "delta_rdt_in_stock", "base": 0.15, "distribution": "Truncated Normal(mean=0.15, sd=0.04)", "range": "[0.05, 0.25]", "source": "Narrowed assumption around RCT effect"},
        {"name": "delta_test_if_stock", "base": 0.19, "distribution": "Truncated Normal(mean=0.19, sd=0.05)", "range": "[0.05, 0.33]", "source": "Narrowed assumption around RCT effect"},
    ],
    "Structural pathway terms held fixed in PSA": [
        {"name": "control_rdt_in_stock", "base": 0.81, "distribution": "Fixed", "range": "fixed", "source": "Revised deterministic base case"},
        {"name": "control_test_if_stock", "base": 0.60, "distribution": "Fixed", "range": "fixed", "source": "Revised deterministic base case"},
        {"name": "control_act_if_positive", "base": 0.82, "distribution": "Fixed", "range": "fixed", "source": "Revised deterministic base case"},
        {"name": "pass_through", "base": 0.82, "distribution": "Fixed", "range": "fixed", "source": "Structural sensitivity handled outside PSA"},
    ],
    "Demography and burden": [
        {"name": "LE0", "base": 62.0, "distribution": "Truncated Normal(mean=62, sd=8)", "range": "[45, 80]", "source": "Narrowed from supplementary methods"},
    ],
    "Cost terms": [
        {"name": "program_cost_scale", "base": 1.0, "distribution": "Removed; represented structurally by 3 scenarios", "range": "fixed", "source": "Revised PSA design"},
        {"name": "medical_cost_scale", "base": 1.0, "distribution": "Gamma(mean=1, CV=0.25)", "range": "positive support", "source": "Tightened from previous PSA"},
    ],
}


In [ ]:
# ---------------------------------------------------------------------
# Deterministic cost schedule for one scenario
# ---------------------------------------------------------------------

def build_cost_schedule(scenario: dict) -> pd.DataFrame:
    rows = []
    prev_tops = 0.0

    for year in range(1, YEARS + 1):
        drones = COMMON_DRONES_BY_YEAR[year - 1]
        tops = COMMON_TOPS_BY_YEAR[year - 1]
        new_tops = tops - prev_tops

        if year == 1:
            flights_per_drone_month = 80.0
            annual_deliveries = 1760.0
            delivery_scale = 1.0
            utilization_scale = 1.0

            program_admin = base_program_admin_y1
            targeting = base_startup_y1
            training = base_training_y1
            monitoring = base_monitoring_y1
            implementation = drone1_rental + drone2_rental + takeoff_point + packing_material
        else:
            flights_per_drone_month = scenario["flights_per_drone_month_year2plus"]
            annual_deliveries = drones * 12.0 * flights_per_drone_month
            delivery_scale = annual_deliveries / 1760.0
            utilization_scale = min(1.0, flights_per_drone_month / 80.0)
            support_multiplier = scenario["support_multiplier_year2plus"]

            program_admin = base_program_admin_y1 * tops * support_multiplier
            targeting = (startup_field_travel + comm_materials) * new_tops * support_multiplier
            training = training_initial * new_tops * support_multiplier
            if year in [3, 6, 9]:
                training += training_refresh * tops * support_multiplier
            monitoring = base_monitoring_y1 * tops * support_multiplier

            overage_flights_per_drone_month = max(
                0.0, flights_per_drone_month - scenario["overage_threshold"]
            )
            implementation = (
                scenario["drone_month_fee"] * 12.0 * drones
                + scenario["top_month_fee"] * 12.0 * tops
                + packing_cost_per_delivery * annual_deliveries
                + scenario["overage_fee"] * overage_flights_per_drone_month * 12.0 * drones
            )

        direct_cost = program_admin + targeting + training + implementation + monitoring
        overhead = 0.10 * direct_cost
        gross_cost = direct_cost + overhead
        offsets = offsets_y1 * delivery_scale
        net_cost = gross_cost - offsets
        net_cost_pv = net_cost / ((1.0 + DISCOUNT_RATE) ** (year - 1))

        rows.append({
            "Year": year,
            "Drones": drones,
            "TOPs": tops,
            "Flights_per_drone_month": flights_per_drone_month,
            "Annual_deliveries": annual_deliveries,
            "Delivery_scale_vs_year1": delivery_scale,
            "Utilisation_scale": utilization_scale,
            "Program_admin": program_admin,
            "Targeting": targeting,
            "Training": training,
            "Implementation": implementation,
            "Monitoring": monitoring,
            "Overhead": overhead,
            "Gross_cost": gross_cost,
            "Offsets": offsets,
            "Net_cost": net_cost,
            "Net_cost_PV": net_cost_pv,
        })
        prev_tops = tops

    df = pd.DataFrame(rows)
    df["Cumulative_net_cost_PV"] = df["Net_cost_PV"].cumsum()
    return df

In [ ]:
# ---------------------------------------------------------------------
# Revised effectiveness model
# ---------------------------------------------------------------------

@dataclass
class TransitionInputs:
    p_HD: float
    p_HU: float
    p_UH: float
    p_US: float
    p_UD: float
    p_SH: float
    p_SD: float
    act_eff: float = 0.96


def service_pathway(
    utilization_scale: float,
    control_rdt_in_stock: float,
    delta_rdt_in_stock: float,
    control_test_if_stock: float,
    delta_test_if_stock: float,
    control_act_if_positive: float,
    pass_through: float,
) -> dict[str, float]:
    stock_control = clamp01(control_rdt_in_stock)
    stock_intervention = clamp01(control_rdt_in_stock + delta_rdt_in_stock * utilization_scale)

    test_control = clamp01(control_test_if_stock)
    test_intervention = clamp01(control_test_if_stock + delta_test_if_stock * utilization_scale)

    p_tested_control = stock_control * test_control
    p_tested_intervention = stock_intervention * test_intervention

    p_prompt_treated_control = p_tested_control * clamp01(control_act_if_positive)
    incremental_testing = max(0.0, p_tested_intervention - p_tested_control)
    p_prompt_treated_intervention = min(
        p_tested_intervention,
        p_prompt_treated_control + incremental_testing * clamp01(pass_through),
    )

    return {
        "P_tested_control": p_tested_control,
        "P_tested_intervention": p_tested_intervention,
        "P_prompt_treated_control": p_prompt_treated_control,
        "P_prompt_treated_intervention": p_prompt_treated_intervention,
    }



def build_transition_matrix(inp: TransitionInputs, p_prompt_treated: float, dt: float = cycle_length) -> np.ndarray:
    P = np.zeros((4, 4), dtype=float)

    healthy_row = rates_to_row_probs({"H->U": prob_to_rate(inp.p_HU), "H->D": prob_to_rate(inp.p_HD)}, dt)
    P[H, H] = healthy_row["stay"]
    P[H, U] = healthy_row["H->U"]
    P[H, D] = healthy_row["H->D"]

    p_UH_treated = inp.p_UH + (inp.p_UD - inp.p_UD * (1.0 - inp.act_eff))
    p_US_treated = inp.p_US
    p_UD_treated = inp.p_UD * (1.0 - inp.act_eff)

    untreated_row = rates_to_row_probs(
        {"U->H": prob_to_rate(inp.p_UH), "U->S": prob_to_rate(inp.p_US), "U->D": prob_to_rate(inp.p_UD)}, dt
    )
    treated_row = rates_to_row_probs(
        {"U->H": prob_to_rate(p_UH_treated), "U->S": prob_to_rate(p_US_treated), "U->D": prob_to_rate(p_UD_treated)}, dt
    )

    q = clamp01(p_prompt_treated)
    P[U, H] = (1.0 - q) * untreated_row["U->H"] + q * treated_row["U->H"]
    P[U, U] = (1.0 - q) * untreated_row["stay"] + q * treated_row["stay"]
    P[U, S] = (1.0 - q) * untreated_row["U->S"] + q * treated_row["U->S"]
    P[U, D] = (1.0 - q) * untreated_row["U->D"] + q * treated_row["U->D"]

    severe_row = rates_to_row_probs({"S->H": prob_to_rate(inp.p_SH), "S->D": prob_to_rate(inp.p_SD)}, dt)
    P[S, H] = severe_row["S->H"]
    P[S, S] = severe_row["stay"]
    P[S, D] = severe_row["S->D"]
    P[D, D] = 1.0
    return P



def run_open_cohort(
    transition_matrices: list[np.ndarray],
    births_per_month_value: float,
    initial_u5_value: float,
    n_age_months: int = 60,
) -> np.ndarray:
    age_weights = np.ones(n_age_months) / n_age_months
    age_state = np.zeros((n_age_months, 4), dtype=float)
    age_state[:, H] = initial_u5_value * age_weights

    trace = np.zeros((len(transition_matrices) + 1, 4), dtype=float)
    trace[0, :] = age_state.sum(axis=0)
    cumulative_deaths_from_aged_out = 0.0

    for t, P in enumerate(transition_matrices):
        age_state = age_state @ P
        aged_out = age_state[-1, :].copy()
        cumulative_deaths_from_aged_out += aged_out[D]

        next_age_state = np.zeros_like(age_state)
        next_age_state[1:, :] = age_state[:-1, :]
        next_age_state[0, H] = births_per_month_value
        age_state = next_age_state

        trace[t + 1, :] = age_state.sum(axis=0)
        trace[t + 1, D] += cumulative_deaths_from_aged_out

    return trace


# Precomputed constants for faster PSA runs
U_WEIGHTS_PLACEHOLDER = np.array([0.0, BASE_PARAMS["u_U"], BASE_PARAMS["u_S"], 0.0], dtype=float)
MEDICAL_COST_BASE = 16.25
OTHER_HEALTH_COSTS = 30.29
MONTHLY_DISCOUNT_10Y = 1.0 / np.power(1.0 + DISCOUNT_RATE * cycle_length, np.arange(n_cycles + 1))
MONTHLY_DISCOUNT_1Y = 1.0 / np.power(1.0 + DISCOUNT_RATE * cycle_length, np.arange(13))
CYCLE_WEIGHTS_10Y = simpson_wcc(n_cycles)
CYCLE_WEIGHTS_1Y = simpson_wcc(12)

SCENARIO_CACHE = {}
for scenario in SCENARIOS:
    schedule = build_cost_schedule(scenario)
    SCENARIO_CACHE[scenario["Scenario_ID"]] = {
        "scenario": scenario,
        "schedule": schedule,
        "util_scales": schedule["Utilisation_scale"].to_numpy(dtype=float),
        "coverage_scale": np.concatenate(
            ([float(schedule.loc[0, "Delivery_scale_vs_year1"])], np.repeat(schedule["Delivery_scale_vs_year1"].to_numpy(dtype=float), 12))
        ),
        "program_cost_monthly": np.concatenate(
            ([0.0], np.repeat(schedule["Net_cost"].to_numpy(dtype=float) / 12.0, 12))
        ),
    }


In [ ]:
# ---------------------------------------------------------------------
# PSA sampling and draw runner
# ---------------------------------------------------------------------

def sample_parameters(rng: np.random.Generator) -> dict[str, float]:
    draw = dict(BASE_PARAMS)

    # Tightened clinical priors
    for name in ["p_HD", "p_HU", "p_UH", "p_US", "p_UD", "p_SH", "p_SD"]:
        a, b = beta_from_mean_k(BASE_PARAMS[name], k=100)
        draw[name] = float(rng.beta(a, b))

    # Tightened disability-weight priors
    for name in ["u_U", "u_S"]:
        a, b = beta_from_mean_k(BASE_PARAMS[name], k=100)
        draw[name] = float(rng.beta(a, b))

    # Narrower trial-effect distributions
    draw["delta_rdt_in_stock"] = truncnorm_sample(0.15, 0.04, 0.05, 0.25, rng)
    draw["delta_test_if_stock"] = truncnorm_sample(0.19, 0.05, 0.05, 0.33, rng)

    # Narrower life-expectancy distribution
    draw["LE0"] = truncnorm_sample(62.0, 8.0, 45.0, 80.0, rng)

    # Program-cost multiplier removed; only residual medical-cost uncertainty retained
    draw["medical_cost_scale"] = gamma_multiplier(mean=1.0, cv=0.25, rng=rng)
    return draw



def run_draw_all_scenarios(params: dict[str, float]) -> list[dict[str, float]]:
    transition_inputs = TransitionInputs(
        p_HD=params["p_HD"],
        p_HU=params["p_HU"],
        p_UH=params["p_UH"],
        p_US=params["p_US"],
        p_UD=params["p_UD"],
        p_SH=params["p_SH"],
        p_SD=params["p_SD"],
        act_eff=params["act_eff"],
    )

    # Control arm is common across all cost scenarios in a given draw.
    control_path = service_pathway(
        utilization_scale=1.0,
        control_rdt_in_stock=params["control_rdt_in_stock"],
        delta_rdt_in_stock=params["delta_rdt_in_stock"],
        control_test_if_stock=params["control_test_if_stock"],
        delta_test_if_stock=params["delta_test_if_stock"],
        control_act_if_positive=params["control_act_if_positive"],
        pass_through=params["pass_through"],
    )
    P_control = build_transition_matrix(transition_inputs, control_path["P_prompt_treated_control"])
    trace_control = run_open_cohort([P_control] * n_cycles, births_per_month, initial_u5_population)

    u_vec = np.array([0.0, params["u_U"], params["u_S"], 0.0], dtype=float)
    c_val = (OTHER_HEALTH_COSTS + MEDICAL_COST_BASE) * params["medical_cost_scale"]
    c_val0 = OTHER_HEALTH_COSTS * params["medical_cost_scale"]
    c_vec = np.array([c_val0, c_val, c_val, 0.0], dtype=float)

    yld_control = trace_control.dot(u_vec) * cycle_length
    new_deaths_control = np.diff(trace_control[:, D], prepend=0.0)
    new_deaths_control[new_deaths_control < 0] = 0.0
    pv_le = (1.0 - np.exp(-DISCOUNT_RATE * params["LE0"])) / DISCOUNT_RATE
    yll_control = new_deaths_control * pv_le
    dalys_control_cycle = yld_control + yll_control
    medical_cost_control_cycle = trace_control.dot(c_vec) * cycle_length

    outputs = []
    for scenario_id, cached in SCENARIO_CACHE.items():
        intervention_matrices: list[np.ndarray] = []
        for util_scale in cached["util_scales"]:
            yearly_path = service_pathway(
                utilization_scale=float(util_scale),
                control_rdt_in_stock=params["control_rdt_in_stock"],
                delta_rdt_in_stock=params["delta_rdt_in_stock"],
                control_test_if_stock=params["control_test_if_stock"],
                delta_test_if_stock=params["delta_test_if_stock"],
                control_act_if_positive=params["control_act_if_positive"],
                pass_through=params["pass_through"],
            )
            intervention_matrices += [
                build_transition_matrix(transition_inputs, yearly_path["P_prompt_treated_intervention"])
            ] * 12

        trace_intervention = run_open_cohort(intervention_matrices, births_per_month, initial_u5_population)

        yld_intervention = trace_intervention.dot(u_vec) * cycle_length
        new_deaths_intervention = np.diff(trace_intervention[:, D], prepend=0.0)
        new_deaths_intervention[new_deaths_intervention < 0] = 0.0
        yll_intervention = new_deaths_intervention * pv_le
        dalys_intervention_cycle = yld_intervention + yll_intervention
        medical_cost_intervention_cycle = trace_intervention.dot(c_vec) * cycle_length

        incremental_dalys_cycle = (dalys_control_cycle - dalys_intervention_cycle) * cached["coverage_scale"]
        incremental_medical_cost_cycle = (medical_cost_intervention_cycle - medical_cost_control_cycle) * cached["coverage_scale"]
        incremental_total_cost_cycle = incremental_medical_cost_cycle + cached["program_cost_monthly"]

        inc_cost_1y = float(np.dot(incremental_total_cost_cycle[:13], MONTHLY_DISCOUNT_1Y * CYCLE_WEIGHTS_1Y))
        inc_cost_10y = float(np.dot(incremental_total_cost_cycle, MONTHLY_DISCOUNT_10Y * CYCLE_WEIGHTS_10Y))
        daly_1y = float(np.dot(incremental_dalys_cycle[:13], MONTHLY_DISCOUNT_1Y * CYCLE_WEIGHTS_1Y))
        daly_10y = float(np.dot(incremental_dalys_cycle, MONTHLY_DISCOUNT_10Y * CYCLE_WEIGHTS_10Y))

        outputs.append({
            "Scenario_ID": scenario_id,
            "Scenario": cached["scenario"]["Scenario"],
            "inc_cost_1y": inc_cost_1y,
            "daly_averted_1y": daly_1y,
            "icer_1y": safe_icer(inc_cost_1y, daly_1y),
            "inc_cost_10y": inc_cost_10y,
            "daly_averted_10y": daly_10y,
            "icer_10y": safe_icer(inc_cost_10y, daly_10y),
        })
    return outputs


In [ ]:
# ---------------------------------------------------------------------
# Figure builders
# ---------------------------------------------------------------------

def save_mean_ce_plane_pdf(daly_draws: np.ndarray, cost_draws: np.ndarray, wtp: float, title: str, path: Path) -> None:
    fig, ax = plt.subplots(figsize=(9, 7))
    ax.grid(True, alpha=0.3)
    ax.scatter(daly_draws, cost_draws, s=10, alpha=0.35)
    mean_x = float(np.mean(daly_draws))
    mean_y = float(np.mean(cost_draws))
    ax.scatter(mean_x, mean_y, s=120, zorder=3)
    ax.annotate("Mean", xy=(mean_x, mean_y), xytext=(6, 6), textcoords="offset points", fontsize=10)
    ax.axhline(0, linewidth=1)
    ax.axvline(0, linewidth=1)
    x0, x1 = ax.get_xlim()
    x_vals = np.array([max(0.0, x0), x1])
    ax.plot(x_vals, wtp * x_vals, linewidth=1)
    ax.set_xlabel("Incremental effectiveness (DALYs averted)")
    ax.set_ylabel("Incremental cost (USD)")
    ax.set_title(title)
    fig.tight_layout()
    fig.savefig(path, format="pdf", bbox_inches="tight")
    plt.close(fig)



def save_ceac_pdf(wtp_thresholds: np.ndarray, ceac: np.ndarray, wtp: float, title: str, path: Path) -> None:
    fig, ax = plt.subplots(figsize=(8, 6))
    ax.plot(wtp_thresholds, ceac)
    ax.axvline(wtp, linestyle="--")
    ax.set_ylim(0, 1)
    ax.set_xlabel("Willingness-to-pay (USD per DALY averted)")
    ax.set_ylabel("Probability cost-effective")
    ax.set_title(title)
    ax.grid(True, alpha=0.3)
    fig.tight_layout()
    fig.savefig(path, format="pdf", bbox_inches="tight")
    plt.close(fig)



def _add_box(ax, xy, width, height, text, fontsize=10):
    x, y = xy
    patch = FancyBboxPatch(
        (x, y), width, height,
        boxstyle="round,pad=0.01,rounding_size=0.01",
        linewidth=1.0,
        facecolor="white",
        edgecolor="black",
    )
    ax.add_patch(patch)
    ax.text(x + width / 2, y + height / 2, text, ha="center", va="center", fontsize=fontsize, wrap=True)
    return patch



def _add_arrow(ax, start, end):
    arrow = FancyArrowPatch(start, end, arrowstyle="->", mutation_scale=12, linewidth=1.0)
    ax.add_patch(arrow)



def save_model_structure_pdf(path: Path) -> None:
    fig, ax = plt.subplots(figsize=(15, 10))
    ax.set_xlim(0, 1)
    ax.set_ylim(0, 1)
    ax.axis("off")

    ax.text(0.02, 0.97, "Revised malaria model structure and PSA setup", fontsize=16, va="top", ha="left")

    _add_box(ax, (0.03, 0.78), 0.24, 0.10, "Trial-derived service pathway\nRDT in stock -> tested -> prompt ACT")
    _add_box(ax, (0.03, 0.62), 0.24, 0.10, "Annual intervention effect\nupdated by realised utilization and delivery reach")
    _add_box(ax, (0.03, 0.46), 0.24, 0.10, "Three deterministic cost scenarios\nexpensive pilot / GAVI transition / at-scale")
    _add_arrow(ax, (0.15, 0.78), (0.15, 0.72))
    _add_arrow(ax, (0.15, 0.62), (0.15, 0.56))

    _add_box(ax, (0.35, 0.80), 0.14, 0.08, "Healthy")
    _add_box(ax, (0.35, 0.66), 0.14, 0.08, "Uncomplicated")
    _add_box(ax, (0.35, 0.52), 0.14, 0.08, "Severe")
    _add_box(ax, (0.35, 0.38), 0.14, 0.08, "Death")
    ax.text(0.42, 0.91, "Age-banded open cohort\nBirths enter monthly; children age out smoothly at 5 years", ha="center", va="center", fontsize=10)
    _add_arrow(ax, (0.42, 0.80), (0.42, 0.74))
    _add_arrow(ax, (0.42, 0.66), (0.42, 0.60))
    _add_arrow(ax, (0.42, 0.52), (0.42, 0.46))
    _add_arrow(ax, (0.27, 0.67), (0.35, 0.70))
    _add_arrow(ax, (0.27, 0.51), (0.35, 0.56))

    _add_box(ax, (0.58, 0.70), 0.16, 0.11, "Outputs per draw\nIncremental cost\nDALYs averted\nICER")
    _add_box(ax, (0.58, 0.52), 0.16, 0.11, "Decision outputs\nCE plane\nCEAC\nWTP thresholds")
    _add_arrow(ax, (0.49, 0.70), (0.58, 0.75))
    _add_arrow(ax, (0.49, 0.56), (0.58, 0.58))
    _add_arrow(ax, (0.66, 0.70), (0.66, 0.63))

    bullet_lines = ["PSA specification", "", "Clinical probabilities", "- Beta(mean, k=100)", "", "Disability weights", "- Beta(mean, k=100)", "", "RCT deltas", "- Narrower truncated Normal", "", "Life expectancy", "- Narrower truncated Normal", "", "Programme costs", "- represented by 3 structural scenarios", "", "Medical costs", "- Gamma(mean=1, CV=0.25)"]
    grouped_text = "\n".join(bullet_lines)
    _add_box(ax, (0.77, 0.36), 0.20, 0.46, grouped_text, fontsize=9)

    ax.text(
        0.03,
        0.24,
        "Base-case PSA notes\n"
        "- Three cost scenarios replace the prior 6-scenario set\n"
        "- ACT pass-through remains fixed and is handled separately as a structural sensitivity analysis\n"
        "- Programme-cost multipliers were removed to avoid double-counting cost uncertainty already represented by scenario structure",
        ha="left",
        va="top",
        fontsize=10,
    )

    fig.tight_layout()
    fig.savefig(path, format="pdf", bbox_inches="tight")
    plt.close(fig)




def save_combined_ce_plane_pdf(draws_df: pd.DataFrame, wtp: float, path: Path) -> None:
    """Combined probabilistic cost-effectiveness plane for the 3 scenarios."""
    colors = {1: "tab:blue", 2: "tab:orange", 3: "tab:green"}
    fig, ax = plt.subplots(figsize=(9, 7))
    ax.grid(True, alpha=0.3)
    for scenario in SCENARIOS:
        sid = scenario["Scenario_ID"]
        sdf = draws_df.loc[draws_df["Scenario_ID"] == sid]
        x = sdf["daly_averted_10y"].to_numpy(dtype=float)
        y = sdf["inc_cost_10y"].to_numpy(dtype=float)
        color = colors[sid]
        ax.scatter(x, y, s=8, alpha=0.15, color=color, label=f"Scenario {sid}", rasterized=True)
        mean_x = float(np.mean(x))
        mean_y = float(np.mean(y))
        ax.scatter(mean_x, mean_y, s=120, color=color, edgecolor='black', linewidth=0.8, zorder=4)
        ax.annotate(f"Mean {sid}", xy=(mean_x, mean_y), xytext=(6, 6), textcoords='offset points', fontsize=9)

    ax.axhline(0, linewidth=1)
    ax.axvline(0, linewidth=1)
    x0, x1 = ax.get_xlim()
    x_vals = np.array([max(0.0, x0), x1])
    ax.plot(x_vals, wtp * x_vals, linestyle='--', linewidth=1, color='black', label=f'WTP = USD {wtp:.0f}/DALY')
    ax.set_xlabel('Incremental effectiveness (DALYs averted, 10-year)')
    ax.set_ylabel('Incremental cost (USD, 10-year PV)')
    ax.set_title('Combined probabilistic cost-effectiveness plane')
    ax.legend(frameon=True)
    fig.tight_layout()
    fig.savefig(path, format='pdf', bbox_inches='tight')
    plt.close(fig)


def save_combined_ceac_pdf(ceac_df: pd.DataFrame, wtp: float, path: Path) -> None:
    """Combined CEAC for the 3 scenarios."""
    colors = {1: "tab:blue", 2: "tab:orange", 3: "tab:green"}
    fig, ax = plt.subplots(figsize=(8, 6))
    for scenario in SCENARIOS:
        sid = scenario["Scenario_ID"]
        sdf = ceac_df.loc[ceac_df["Scenario_ID"] == sid]
        ax.plot(sdf["WTP_USD_per_DALY"], sdf["Probability_cost_effective"], linewidth=2, color=colors[sid], label=f"Scenario {sid}")
    ax.axvline(wtp, linestyle='--', color='black', linewidth=1)
    ax.set_ylim(0, 1)
    ax.set_xlabel('Willingness-to-pay (USD per DALY averted)')
    ax.set_ylabel('Probability cost-effective')
    ax.set_title('Combined cost-effectiveness acceptability curves')
    ax.grid(True, alpha=0.3)
    ax.legend(frameon=True)
    fig.tight_layout()
    fig.savefig(path, format='pdf', bbox_inches='tight')
    plt.close(fig)


In [ ]:
# ---------------------------------------------------------------------
# Main PSA execution
# ---------------------------------------------------------------------

def main(n_draws: int = 10_000, wtp: float = 540.0) -> None:
    rng = np.random.default_rng(SEED)
    bootstrap_rng = np.random.default_rng(SEED + 100)

    # Run all draws across the 3 retained scenarios
    draw_rows: list[dict[str, float]] = []
    for draw_idx in range(1, n_draws + 1):
        params = sample_parameters(rng)
        scenario_outputs = run_draw_all_scenarios(params)
        for out in scenario_outputs:
            draw_rows.append({
                "draw": draw_idx,
                "Scenario_ID": out["Scenario_ID"],
                "Scenario": out["Scenario"],
                "inc_cost_10y": out["inc_cost_10y"],
                "daly_averted_10y": out["daly_averted_10y"],
            })
        if draw_idx % 1000 == 0 or draw_idx == n_draws:
            print(f"Completed {draw_idx}/{n_draws} draws", flush=True)

    draws_df = pd.DataFrame(draw_rows)

    # Table 3: 10-year PSA summary results for the 3 scenarios
    summary_rows = []
    ceac_rows = []
    wtp_thresholds = np.linspace(0, 3000, 600)
    n_boot = 1000

    for scenario in SCENARIOS:
        sid = scenario["Scenario_ID"]
        sdf = draws_df.loc[draws_df["Scenario_ID"] == sid].copy()
        cost_draws = sdf["inc_cost_10y"].to_numpy(dtype=float)
        eff_draws = sdf["daly_averted_10y"].to_numpy(dtype=float)

        mean_cost, lo_cost, hi_cost = quantile_summary(cost_draws)
        mean_eff, lo_eff, hi_eff = quantile_summary(eff_draws)

        icer_point = safe_icer(float(np.mean(cost_draws)), float(np.mean(eff_draws)))
        boot_icers = []
        for _ in range(n_boot):
            idx = bootstrap_rng.integers(0, len(sdf), size=len(sdf))
            boot_icers.append(safe_icer(float(np.mean(cost_draws[idx])), float(np.mean(eff_draws[idx]))))
        boot_icers = np.asarray(boot_icers, dtype=float)
        boot_icers = boot_icers[np.isfinite(boot_icers)]
        icer_lo = float(np.percentile(boot_icers, 2.5)) if boot_icers.size else np.nan
        icer_hi = float(np.percentile(boot_icers, 97.5)) if boot_icers.size else np.nan

        positive_effect = np.maximum(eff_draws, 0.0)
        ce_mask = (cost_draws <= wtp * positive_effect) & (positive_effect > 0)
        successes = int(np.sum(ce_mask))
        prob_ce = 100.0 * successes / len(sdf)
        wil_lo, wil_hi = wilson_ci(successes, len(sdf))

        is_ce = (cost_draws[None, :] <= wtp_thresholds[:, None] * positive_effect[None, :]) & (positive_effect[None, :] > 0)
        ceac = is_ce.mean(axis=1)
        for thresh, prob in zip(wtp_thresholds, ceac):
            ceac_rows.append({
                "Scenario_ID": sid,
                "Scenario": scenario["Scenario"],
                "WTP_USD_per_DALY": float(thresh),
                "Probability_cost_effective": float(prob),
            })

        summary_rows.append({
            "Scenario_ID": sid,
            "Scenario": scenario["Scenario"],
            "Mean_incremental_cost_10y_USD": mean_cost,
            "Lower_95_CI_incremental_cost_10y_USD": lo_cost,
            "Upper_95_CI_incremental_cost_10y_USD": hi_cost,
            "Mean_DALYs_averted_10y": mean_eff,
            "Lower_95_CI_DALYs_averted_10y": lo_eff,
            "Upper_95_CI_DALYs_averted_10y": hi_eff,
            "Mean_ICER_10y_USD_per_DALY": icer_point,
            "Lower_95_CI_ICER_10y_USD_per_DALY": icer_lo,
            "Upper_95_CI_ICER_10y_USD_per_DALY": icer_hi,
            "Probability_cost_effective_at_WTP_540_percent": prob_ce,
            "Lower_95_CI_probability_cost_effective_at_WTP_540_percent": 100.0 * wil_lo,
            "Upper_95_CI_probability_cost_effective_at_WTP_540_percent": 100.0 * wil_hi,
        })

    table3 = pd.DataFrame(summary_rows).sort_values("Scenario_ID").reset_index(drop=True)
    ceac_df = pd.DataFrame(ceac_rows)

    table3_path = "Table_3_PSA_summary_results_10y.csv"
    figure2_path = "Figure_2_combined_probabilistic_CE_plane.pdf"
    figure3_path = "Figure_3_combined_CEAC.pdf"

    table3.to_csv(table3_path, index=False)
    save_combined_ce_plane_pdf(draws_df, wtp, figure2_path)
    save_combined_ceac_pdf(ceac_df, wtp, figure3_path)
# ---------------------------------------------------------
    # NEW: Separate Table for 50%, 80%, and 90% WTP Thresholds
    # ---------------------------------------------------------
    threshold_rows = []

    # Loop through the scenarios stored in the ceac_df
    for sid in ceac_df["Scenario_ID"].unique():
        sdf = ceac_df[ceac_df["Scenario_ID"] == sid]
        scenario_name = sdf["Scenario"].iloc[0]

        # Extract arrays for the interpolation function
        wtps = sdf["WTP_USD_per_DALY"].to_numpy(dtype=float)
        probs = sdf["Probability_cost_effective"].to_numpy(dtype=float)

        wtp_50 = find_wtp_at_probability(wtps, probs, 0.50)
        wtp_80 = find_wtp_at_probability(wtps, probs, 0.80)
        wtp_90 = find_wtp_at_probability(wtps, probs, 0.90)

        threshold_rows.append({
            "Scenario_ID": sid,
            "Scenario": scenario_name,
            "WTP_for_50_percent_CE_USD": wtp_50,
            "WTP_for_80_percent_CE_USD": wtp_80,
            "WTP_for_90_percent_CE_USD": wtp_90
        })

    table4_thresholds = pd.DataFrame(threshold_rows)
    table4_path = "Table_4_WTP_Thresholds.csv"
    table4_thresholds.to_csv(table4_path, index=False)

    # Add to your print statements
    print(f"Wrote: {table4_path}")
    print("\nTable 4: WTP Thresholds Preview:")
    print(table4_thresholds.to_string(index=False))
    print(f"Seed: {SEED}")
    print(f"Wrote: {table3_path}")
    print(f"Wrote: {figure2_path}")
    print(f"Wrote: {figure3_path}")
    print()
    print("Table 3 preview:")
    print(table3.to_string(index=False))


if __name__ == "__main__":
    main(n_draws=10_000, wtp=540.0)


Completed 1000/10000 draws
Completed 2000/10000 draws
Completed 3000/10000 draws
Completed 4000/10000 draws
Completed 5000/10000 draws
Completed 6000/10000 draws
Completed 7000/10000 draws
Completed 8000/10000 draws
Completed 9000/10000 draws
Completed 10000/10000 draws
Wrote: Table_4_WTP_Thresholds.csv

Table 4: WTP Thresholds Preview:
 Scenario_ID                                         Scenario  WTP_for_50_percent_CE_USD  WTP_for_80_percent_CE_USD  WTP_for_90_percent_CE_USD
           1      Expensive pilot pricing + underused network                1557.095159                        NaN                        NaN
           2 Transitional GAVI pricing + expected utilisation                 618.135489                2111.018364                        NaN
           3            At-scale pricing + efficient scale-up                 312.264627                1065.347007                2368.948247
Seed: 12345
Wrote: Table_3_PSA_summary_results_10y.csv
Wrote: Figure_2_combined_probabili